In [1]:
# Import Libraries
import numpy as np
import pandas as pd
import netCDF4 as nc
import xarray as xr
from pathlib import Path
from datetime import datetime, timezone

# Set paths
DATA_DIR = Path("../data/2024-06-14/cleaned")
OUT_FILE = Path("../data/2024-06-14/MB_spireberg_cleaned_20240614_194448.nc")

# Array to contain sounding file column headers
COLUMNS = [
    "Beam", "Classification", "Date Time",
    "Footprint X", "Footprint Y", "Footprint Z",
    "Heave", "Intensity", "KP", "Ping Counter", "Ping Number",
    "Quality", "Sound Speed", "Source Identifier", "System ID", "Tide",
    "Transducer Heading", "Transducer Pitch", "Transducer Roll",
    "Transducer X", "Transducer Y", "Transducer Z",
    "Two Way Travel Time", "Uncertainty Horizontal", "Uncertainty Vertical",
    "Unix Time",
    "Vessel Heading", "Vessel Pitch", "Vessel Roll",
    "Vessel X", "Vessel Y", "Vessel Z",
]

FILES = sorted(DATA_DIR.glob("*.txt"))
N_BEAMS = 1024
N_SEGMENTS = len(FILES)

# CF-1.8 standard fill values
FILL_F32 = np.float32(9.96921e+36)
FILL_F64 = np.float64(9.969209968386869e+36)
FILL_I8  = np.int8(-127)
FILL_I32 = np.int32(-2147483647)


In [ ]:
# Convert DMS to Decimal Degrees
def dms_to_decimal(series):

    # Break input into discrete parts
    parts = series.str.strip().str.split(expand=True)
    #   parts[0] = degrees, parts[1] = minutes, parts[2] = decimal seconds, parts[3] = hemisphere

    # Use DMS numeric data to convert to decimal degrees
    val  = parts[0].astype(float) + parts[1].astype(float)/60 + parts[2].astype(float)/3600

    # Get sign information from NSWE directionality
    sign = parts[3].map({"N": 1.0, "S": -1.0, "E": 1.0, "W": -1.0})

    return (val * sign).values  # Returns a numpy array


In [3]:
# Establish array to hold number of pings per file
ping_counts = []

# For each swath file...
for f in FILES:

    # Open file
    df = pd.read_csv(f, names=COLUMNS, comment="#")

    # Get ping counts from number of entires in ping column
    ping_counts.append(int(df["Ping Number"].nunique()))

    print(f"{f.name}: {ping_counts[-1]:,} pings")

# Calculate total number of pings between all files
total_pings = sum(ping_counts)

# segment_starts[i] = index of the first ping (in the combined array) that came from file i
segment_starts = np.array([sum(ping_counts[:i]) for i in range(N_SEGMENTS)], dtype=np.int32)
segment_counts = np.array(ping_counts, dtype=np.int32)

print(f"\nTotal: {total_pings:,} pings (2-D shape will be ({total_pings}, {N_BEAMS}))")


20240614_194448.txt: 620 pings
20240614_200001.txt: 413 pings
20240614_200947.txt: 1,489 pings
20240614_201828.txt: 1,564 pings
20240614_202551.txt: 338 pings

Total: 4,424 pings (2-D shape will be (4424, 1024))


In [4]:
# Set up arrays to store per-ping and per-sounding metadata

P = total_pings
PB = (total_pings, N_BEAMS)

# Per-ping 1-D arrays 
time_arr = np.full(P, FILL_F64, dtype=np.float64)
ping_num_arr = np.full(P, FILL_I32, dtype=np.int32)
ping_seg_arr = np.full(P, FILL_I8,  dtype=np.int8)

vessel_lon_arr = np.full(P, FILL_F32, dtype=np.float32)
vessel_lat_arr  = np.full(P, FILL_F32, dtype=np.float32)
vessel_z_arr = np.full(P, FILL_F32, dtype=np.float32)
vessel_hdg_arr = np.full(P, FILL_F32, dtype=np.float32)
vessel_pitch_arr = np.full(P, FILL_F32, dtype=np.float32)
vessel_roll_arr = np.full(P, FILL_F32, dtype=np.float32)

tx_lon_arr = np.full(P, FILL_F32, dtype=np.float32)
tx_lat_arr = np.full(P, FILL_F32, dtype=np.float32)
tx_z_arr = np.full(P, FILL_F32, dtype=np.float32)
tx_hdg_arr = np.full(P, FILL_F32, dtype=np.float32)
tx_pitch_arr = np.full(P, FILL_F32, dtype=np.float32)
tx_roll_arr = np.full(P, FILL_F32, dtype=np.float32)

heave_arr = np.full(P, FILL_F32, dtype=np.float32)
tide_arr = np.full(P, FILL_F32, dtype=np.float32)
kp_arr = np.full(P, FILL_F32, dtype=np.float32)
ssp_arr = np.full(P, FILL_F32, dtype=np.float32)
src_id_arr = np.full(P, FILL_I8,  dtype=np.int8)
sys_id_arr = np.full(P, FILL_I8,  dtype=np.int8)

# Per-sounding 2-D arrays
fp_lon_arr = np.full(PB, FILL_F32, dtype=np.float32)
fp_lat_arr = np.full(PB, FILL_F32, dtype=np.float32)
depth_arr = np.full(PB, FILL_F32, dtype=np.float32)
intens_arr = np.full(PB, FILL_F32, dtype=np.float32)
twtt_arr = np.full(PB, FILL_F32, dtype=np.float32)
unc_h_arr = np.full(PB, FILL_F32, dtype=np.float32)
unc_v_arr = np.full(PB, FILL_F32, dtype=np.float32)
quality_arr = np.full(PB, FILL_I8,  dtype=np.int8)
class_arr = np.full(PB, FILL_I8,  dtype=np.int8)

# Number of bytes per array in MBs
mb_2d = fp_lon_arr.nbytes / 1e6


In [5]:
print("Filling arrays …")
ping_offset = 0

# For each swath file...
for seg, f in enumerate(FILES):

    # Open file
    df = pd.read_csv(f, names=COLUMNS, comment="#")

    # Decode DMS columns to decimal degrees
    df["fp_lon"] = dms_to_decimal(df["Footprint X"])
    df["fp_lat"] = dms_to_decimal(df["Footprint Y"])
    df["tx_lon"] = dms_to_decimal(df["Transducer X"])
    df["tx_lat"] = dms_to_decimal(df["Transducer Y"])
    df["vessel_lon"] = dms_to_decimal(df["Vessel X"])
    df["vessel_lat"] = dms_to_decimal(df["Vessel Y"])

    # Map Ping Number to a sorted 0-based local index
    local_idx, unique_pings = pd.factorize(df["Ping Number"], sort=True)
    local_idx = local_idx.astype(np.int32)
    n_pings = len(unique_pings)
    global_idx = ping_offset + local_idx 
    beam_idx   = df["Beam"].values.astype(np.int32) 

    # Fill two dimensional array where rows are pings and columns are beams

    # Geographic position of beam measurement
    fp_lon_arr[global_idx, beam_idx] = df["fp_lon"].values.astype(np.float32)
    fp_lat_arr[global_idx, beam_idx] = df["fp_lat"].values.astype(np.float32)
    depth_arr[global_idx, beam_idx] = df["Footprint Z"].values.astype(np.float32)

    # Intensity of return 
    intens_arr[global_idx, beam_idx] = df["Intensity"].values.astype(np.float32)

    # Time of flight
    twtt_arr[global_idx, beam_idx] = df["Two Way Travel Time"].values.astype(np.float32)

    # Uncertainty 
    unc_h_arr[global_idx, beam_idx] = df["Uncertainty Horizontal"].values.astype(np.float32)
    unc_v_arr[global_idx, beam_idx] = df["Uncertainty Vertical"].values.astype(np.float32)

    # Quality
    quality_arr[global_idx, beam_idx] = df["Quality"].values.astype(np.int8)

    # Classification
    class_arr[global_idx, beam_idx] = df["Classification"].values.astype(np.int8)

    # Fill one dimensional array with data for each ping

    # Retrieve metadata from first row of soundings from a unique ping
    first = df.groupby("Ping Number").first().loc[unique_pings]

    # Establish where pings will be stored in cumulative array for file currently being processed
    sl = slice(ping_offset, ping_offset + n_pings)

    # Time
    time_arr[sl] = first["Unix Time"].values

    # Ping number 
    ping_num_arr[sl] = unique_pings.astype(np.int32)

    # Swath number
    ping_seg_arr[sl] = np.int8(seg)

    # Vessel coordinates
    vessel_lon_arr[sl] = first["vessel_lon"].values.astype(np.float32)
    vessel_lat_arr[sl] = first["vessel_lat"].values.astype(np.float32)
    vessel_z_arr[sl] = first["Vessel Z"].values.astype(np.float32)

    # Vessel orientation
    vessel_hdg_arr[sl] = first["Vessel Heading"].values.astype(np.float32)
    vessel_pitch_arr[sl] = first["Vessel Pitch"].values.astype(np.float32)
    vessel_roll_arr[sl] = first["Vessel Roll"].values.astype(np.float32)

    # Transducer coordinates
    tx_lon_arr[sl] = first["tx_lon"].values.astype(np.float32)
    tx_lat_arr[sl] = first["tx_lat"].values.astype(np.float32)
    tx_z_arr[sl] = first["Transducer Z"].values.astype(np.float32)

    # Transducer orientation
    tx_hdg_arr[sl] = first["Transducer Heading"].values.astype(np.float32)
    tx_pitch_arr[sl] = first["Transducer Pitch"].values.astype(np.float32)
    tx_roll_arr[sl] = first["Transducer Roll"].values.astype(np.float32)

    # Other vessel parameters
    heave_arr[sl] = first["Heave"].values.astype(np.float32)
    tide_arr[sl] = first["Tide"].values.astype(np.float32)
    kp_arr[sl] = first["KP"].values.astype(np.float32)

    # Sound speed from SVP
    ssp_arr[sl] = first["Sound Speed"].values.astype(np.float32)

    # MBES metadata
    src_id_arr[sl] = first["Source Identifier"].values.astype(np.int8)
    sys_id_arr[sl] = first["System ID"].values.astype(np.int8)

    # Record ping offset in array for processing of next file
    ping_offset += n_pings
    print(f"[{seg+1}/{N_SEGMENTS}] {f.name} rows {sl.start}–{sl.stop-1}")

print("Done.")


Filling arrays …
[1/5] 20240614_194448.txt rows 0–619
[2/5] 20240614_200001.txt rows 620–1032
[3/5] 20240614_200947.txt rows 1033–2521
[4/5] 20240614_201828.txt rows 2522–4085
[5/5] 20240614_202551.txt rows 4086–4423
Done.


In [6]:
# Compression parameters
COMP = {"zlib": True, "complevel": 4}
CHUNK_P = (min(512, total_pings),)
CHUNK_PB = (min(128, total_pings), N_BEAMS)

# File creation datetime
created = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Set output directory
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with nc.Dataset(OUT_FILE, "w", format="NETCDF4") as ds:

    # NetCDF Metadata
    ds.Conventions = "CF-1.8"
    ds.title = "MBES Survey 1 — Spireberg, LeConte Bay, Alaska — 2024-06-14"
    ds.source = "NORBIT Multibeam Sonar deployed from the Polly Uncrewed Surface Vessel (cleaned in Qimera)"
    ds.history = f"Created {created}"
    ds.featureType = "swath"
    ds.geospatial_lon_min = float(np.min(fp_lon_arr[fp_lon_arr != FILL_F32]))
    ds.geospatial_lon_max = float(np.max(fp_lon_arr[fp_lon_arr != FILL_F32]))
    ds.geospatial_lat_min = float(np.min(fp_lat_arr[fp_lat_arr != FILL_F32]))
    ds.geospatial_lat_max = float(np.max(fp_lat_arr[fp_lat_arr != FILL_F32]))
    ds.time_coverage_start = str(np.min(time_arr[time_arr != FILL_F64]))
    ds.time_coverage_end = str(np.max(time_arr[time_arr != FILL_F64]))

    # Dimensions
    ds.createDimension("ping", total_pings)
    ds.createDimension("beam", N_BEAMS)
    ds.createDimension("segment", N_SEGMENTS)

    # CRS
    crs = ds.createVariable("crs", "i1")
    crs.grid_mapping_name = "latitude_longitude"
    crs.longitude_of_prime_meridian = 0.0
    crs.semi_major_axis = 6378137.0
    crs.inverse_flattening = 298.257222101
    crs.crs_wkt = "EPSG:6318" # NAD83(2011) horizontal
    crs.vertical_datum = "EPSG:5703" # NAVD88 vertical

    # Beam coordinate
    v = ds.createVariable("beam", "i2", ("beam",))
    v[:] = np.arange(N_BEAMS, dtype=np.int16)
    v.long_name = "Beam index (0 - 1023) based on transducer beam was emitted from"
    v.units = "1"

    # Source file name
    v = ds.createVariable("source_file", str, ("segment",))
    for i, f in enumerate(FILES):
        v[i] = f.name
    v.long_name = "Source Filename"

    # Timestamp
    v = ds.createVariable("time", "f8", ("ping",), fill_value=FILL_F64,
                          **COMP, chunksizes=CHUNK_P)
    v[:] = time_arr
    v.standard_name = "Time"
    v.long_name     = "Ping Time"
    v.units         = "seconds since 1970-01-01T00:00:00Z"
    v.calendar      = "Gregorian"

    # Per-ping metadata
    v = ds.createVariable("ping_num", "i4", ("ping",), fill_value=FILL_I32, **COMP, chunksizes=CHUNK_P)
    v[:] = ping_num_arr
    v.long_name = "Ping Index (within swath file)"

    # Swath metadata
    v = ds.createVariable("ping_segment", "i1", ("ping",), fill_value=FILL_I8, **COMP, chunksizes=CHUNK_P)
    v[:] = ping_seg_arr
    v.long_name   = "Swath file index"
    v.valid_range = np.array([0, N_SEGMENTS - 1], dtype=np.int8)

    # Vessel & transducer state
    _per_ping_f32 = [
        ("vessel_lon", vessel_lon_arr, "Vessel Longitude", "degrees_east"),
        ("vessel_lat", vessel_lat_arr, "Vessel Latitude", "degrees_north"),
        ("vessel_z", vessel_z_arr, "Vessel Depth (Negative Down)", "m"),
        ("vessel_heading", vessel_hdg_arr, "Vessel True Heading", "degree"),
        ("vessel_pitch", vessel_pitch_arr, "Vessel Pitch", "degree"),
        ("vessel_roll", vessel_roll_arr, "Vessel Roll", "degree"),
        ("transducer_lon", tx_lon_arr, "Transducer Longitude", "degrees_east"),
        ("transducer_latitude", tx_lat_arr, "Transducer Latitude", "degrees_north"),
        ("transducer_z", tx_z_arr, "Transducer Depth (negative down)", "m"),
        ("transducer_heading", tx_hdg_arr, "Transducer Heading", "degree"),
        ("transducer_pitch", tx_pitch_arr, "Transducer Pitch", "degree"),
        ("transducer_roll", tx_roll_arr, "Transducer Roll", "degree"),
        ("heave", heave_arr, "Heave", "m"),
        ("tide", tide_arr, "Tide Correction", "m"),
        ("kp", kp_arr, "Along-track Kilometer Point", "m"),
        ("sound_speed", ssp_arr, "Surface Sound Speed (m/s)", "m s-1"),
    ]

    # CRS
    for name, arr, lname, units in _per_ping_f32:
        v = ds.createVariable(name, "f4", ("ping",), fill_value=FILL_F32, **COMP, chunksizes=CHUNK_P)
        v[:] = arr
        v.long_name = lname
        v.units = units
        v.grid_mapping = "crs"

    # Source ID
    v = ds.createVariable("source_identifier", "i1", ("ping",), fill_value=FILL_I8, **COMP, chunksizes=CHUNK_P)
    v[:] = src_id_arr
    v.long_name = "Source Identifier"

    # MBES System ID
    v = ds.createVariable("system_id", "i1", ("ping",), fill_value=FILL_I8, **COMP, chunksizes=CHUNK_P)
    v[:] = sys_id_arr
    v.long_name = "System ID"

    # Per-sounding 2-D variables
    _per_sounding = [
        ("footprint_longitude", fp_lon_arr,  "Sounding Footprint Longitude", "degrees_east"),
        ("footprint_latitude", fp_lat_arr, "Sounding Footprint Latitude", "degrees_north"),
        ("depth", depth_arr, "Sounding Depth (negative down)", "m"),
        ("intensity", intens_arr, "Backscatter Intensity", "1"),
        ("two_way_travel_time", twtt_arr, "Two-way Travel Time", "s"),
        ("uncertainty_horizontal", unc_h_arr, "Horizontal Positional Uncertainty", "m"),
        ("uncertainty_vertical", unc_v_arr, "Vertical Positional Uncertainty", "m"),
    ]

    # CRS
    for name, arr, lname, units in _per_sounding:
        v = ds.createVariable(name, "f4", ("ping", "beam"), fill_value=FILL_F32, **COMP, chunksizes=CHUNK_PB)
        v[:] = arr
        v.long_name = lname
        v.units = units
        v.grid_mapping = "crs"

    # Beam Quality
    v = ds.createVariable("quality", "i1", ("ping", "beam"), fill_value=FILL_I8, **COMP, chunksizes=CHUNK_PB)
    v[:] = quality_arr
    v.long_name = "Beam Quality Flag"

    # Classification
    v = ds.createVariable("classification", "i1", ("ping", "beam"), fill_value=FILL_I8, **COMP, chunksizes=CHUNK_PB)
    v[:] = class_arr
    v.long_name = "Sounding Classification"

size_mb = OUT_FILE.stat().st_size / 1e6
print(f"\nWrote {OUT_FILE} ({size_mb:.1f} MB)")


Wrote ../data/2024-06-14/MB_spireberg_cleaned_20240614_194448.nc (16.1 MB)


In [7]:
xr.open_dataset(OUT_FILE)

<xarray.Dataset> Size: 163MB
Dimensions:                 (segment: 5, ping: 4424, beam: 1024)
Coordinates:
  * beam                    (beam) int16 2kB 0 1 2 3 4 ... 1020 1021 1022 1023
Dimensions without coordinates: segment, ping
Data variables: (12/32)
    crs                     int8 1B ...
    source_file             (segment) <U19 380B ...
    time                    (ping) datetime64[ns] 35kB ...
    ping_num                (ping) float64 35kB ...
    ping_segment            (ping) float32 18kB ...
    vessel_lon              (ping) float32 18kB ...
    ...                      ...
    intensity               (ping, beam) float32 18MB ...
    two_way_travel_time     (ping, beam) float32 18MB ...
    uncertainty_horizontal  (ping, beam) float32 18MB ...
    uncertainty_vertical    (ping, beam) float32 18MB ...
    quality                 (ping, beam) float32 18MB ...
    classification          (ping, beam) float32 18MB ...
Attributes:
    Conventions:          CF-1.8
    title:                MBES Survey 1 — Spireberg, LeConte Bay, Alaska — 20...
    source:               NORBIT Multibeam Sonar deployed from the Polly Uncr...
    history:              Created 2026-06-26T06:19:08+00:00
    featureType:          swath
    geospatial_lon_min:   -132.4110870361328
    geospatial_lon_max:   -132.4096221923828
    geospatial_lat_min:   56.82209014892578
    geospatial_lat_max:   56.823211669921875
    time_coverage_start:  1718393959.555
    time_coverage_end:    1718396496.808